# Homework 4 Solutions

1. Download Daily Bars for FB, AAPL, AMZN, NFLX, GOOGL and QQQ from yahoo finance starting 2016-01-01. Use the Adj Close to compute daily returns.

In [22]:
import yfinance as yf
import numpy as np
import pandas as pd

univ = ['FB','AAPL','AMZN','NFLX','GOOGL','QQQ']
px = yf.download(univ, start="2016-01-01")
ret = px['Close'].pct_change().dropna()

[*********************100%***********************]  6 of 6 completed


2. Now, let's compute the beta of FB, AAPL, AMZN, NFLX, GOOGL using QQQ as our benchmark. You can think of this as the beta these stocks have to their industry (tech). In practice,  we have to use some lookback window to compute the beta. Let's use 252 (1 year, excluding wknds/holidays). So, for each day, the betas should be computed using the most recent 252 data points.

While it's possible to use a loop + statsmodels OLS function here, the computation is simple enough we don't need that complexity and overhead. The formula for beta in a univariate regression (with a constant) is correlation*(vol_y/vol_benchmark). Here I computed this quickly using the pandas rolling function.

In [23]:
corr = ret.rolling(252).corr(ret['QQQ'])
vol = ret.rolling(252).std()
beta = (corr*vol).divide(vol['QQQ'],axis=0)  

3. Using the betas, compute an "alpha" on each day. This is also known as a "residual return".

In [24]:
resid = ret - beta.multiply(ret['QQQ'],0)

4. Compare the volatility of the residual returns to that of the original returns. What do you notice?

The residual returns generally have much lower volatility. This is because we've taken out the "beta component" which drives much of the stock returns.

In [25]:
vol = {}
vol['orig'] = ret.std()*np.sqrt(252)
vol['resid'] = resid.std()*np.sqrt(252)
vol = pd.DataFrame(vol).drop('QQQ')
vol

,orig,resid
Ticker,,
AAPL,0.217703,NaN
AMZN,0.311494,NaN
FB,0.049430,NaN
GOOGL,0.286791,NaN
NFLX,0.308456,NaN


5. Compare the pairwise correlations of the residual returns to that of the original returns. What do you notice?

The pairwise correlations of the residual returns in general are much lower between stocks. This is because we've taken out one of the major common forces (tides) moving these stocks. Also striking is the large drop in correlations with QQQ. The original returns are 0.6-0.8 correlated with QQQ, but the residual returns are nearly 0 correlated with QQQ. Note, however, that they are not perfectly 0 correlated. This is because in practice we have to estimate the beta using rolling windows rather than the full sample

In [29]:
ret

Ticker,AAPL,AMZN,FB,GOOGL,NFLX,QQQ
Date,,,,,,
2025-06-27,0.000398,0.028464,0.008494,0.028754,0.012589,0.003424
2025-06-30,0.020340,-0.017510,0.005590,-0.012883,0.012100,0.006477
2025-07-01,0.012916,0.004877,-0.001977,-0.002213,-0.034000,-0.008429
2025-07-02,0.022231,-0.002449,-0.003565,0.015924,-0.006756,0.006965
2025-07-03,0.005225,0.015869,0.006559,0.004982,0.009589,0.009840
...,...,...,...,...,...,...
2025-12-16,0.001824,0.000090,-0.000475,-0.005353,0.008532,0.001982
2025-12-17,-0.010087,-0.005796,-0.001545,-0.032130,0.002326,-0.018537
2025-12-18,0.001288,0.024811,0.002809,0.019345,-0.008334,0.014490


In [26]:
ret.corr()

Ticker,AAPL,AMZN,FB,GOOGL,NFLX,QQQ
Ticker,,,,,,
AAPL,1.000000,0.305047,0.280191,0.360607,0.245091,0.482588
AMZN,0.305047,1.000000,0.428174,0.297874,0.220334,0.597255
FB,0.280191,0.428174,1.000000,0.327734,0.126807,0.641171
GOOGL,0.360607,0.297874,0.327734,1.000000,0.076235,0.572616
NFLX,0.245091,0.220334,0.126807,0.076235,1.000000,0.259839
QQQ,0.482588,0.597255,0.641171,0.572616,0.259839,1.000000


In [27]:
resid.corr()

Ticker,AAPL,AMZN,FB,GOOGL,NFLX,QQQ
Ticker,,,,,,
AAPL,NaN,NaN,NaN,NaN,NaN,NaN
AMZN,NaN,NaN,NaN,NaN,NaN,NaN
FB,NaN,NaN,NaN,NaN,NaN,NaN
GOOGL,NaN,NaN,NaN,NaN,NaN,NaN
NFLX,NaN,NaN,NaN,NaN,NaN,NaN
QQQ,NaN,NaN,NaN,NaN,NaN,NaN


6. Compute the information ratio for each of these stocks and compare that to the sharpe ratio

The IRs are in general lower than the SRs, suggesting most of the performance of these stocks was driven by their exposure to QQQ. AAPL performed the best looking at IR or SR and FB performed the worst. 

In [28]:
df = {}
df['IR'] = resid.mean() / resid.std()*np.sqrt(252)
df['SR'] = ret.mean() / ret.std()*np.sqrt(252)
df = pd.DataFrame(df).drop('QQQ')
df

,IR,SR
Ticker,,
AAPL,NaN,2.956831
AMZN,NaN,0.463269
FB,NaN,2.549004
GOOGL,NaN,4.205156
NFLX,NaN,-2.056479
